In [ ]:
import torch
from diffusers import AutoPipelineForInpainting
from diffusers.utils import load_image, make_image_grid

pipeline = AutoPipelineForInpainting.from_pretrained(
    "runwayml/stable-diffusion-inpainting", torch_dtype=torch.float16, variant="fp16"
)
pipeline.enable_model_cpu_offload()

# load base and mask image
init_image = load_image(
    "../data/labeled/images/B-7_default_mmrsbpb_mMRS-BPB_15122023134505.jpg"
)
mask_image = load_image("../mask.png")

init_image_downscaled = init_image.resize((512, 512), resample=3)
mask_image_downscaled = mask_image.resize((512, 512), resample=3)

# generator = torch.Generator("cuda").manual_seed(92)
prompt = "photo of an empty petri dish"
negative_prompt = "bacteria colony, bacterial colonies, bacteria, circles, dots"
image = pipeline(
    prompt=prompt,
    strength=1.0,
    negative_prompt=negative_prompt,
    image=init_image_downscaled,
    mask_image=mask_image_downscaled,
    # num_images_per_prompt=4,
    # generator=generator,
).images[0]

# Upscale the image to the original size
image_upscaled = image.resize(init_image.size, resample=3)

make_image_grid([init_image, mask_image, image_upscaled], rows=1, cols=3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def overlay_mask(image, mask):
    mask = np.array(mask)
    return np.where(mask, image, 255)


# Show mask on top of the result image
overlayed_image = overlay_mask(np.array(image_upscaled), np.array(mask_image))

plt.imshow(overlayed_image)
plt.show()